In [1]:
import sys
print(sys.executable)

c:\Users\prabh\OneDrive\Desktop\FakeJobBERT\.venv\Scripts\python.exe


In [2]:
from datasets import load_dataset

ds = load_dataset("james-burton/fake_job_postings2_ord")
print(ds)

split = list(ds.keys())[0]
print("Split:", split)
print("Columns:", ds[split].column_names)

ds[split][0]

README.md:   0%|          | 0.00/706 [00:00<?, ?B/s]

c:\Users\prabh\OneDrive\Desktop\FakeJobBERT\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prabh\.cache\huggingface\hub\datasets--james-burton--fake_job_postings2_ord. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001-514dfbbc908283(…):   0%|          | 0.00/8.23M [00:00<?, ?B/s]

data/validation-00000-of-00001-1690ecf9b(…):   0%|          | 0.00/1.41M [00:00<?, ?B/s]

data/test-00000-of-00001-8d95ffc75200019(…):   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10816 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1909 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3182 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'salary_range', 'description', 'required_experience', 'required_education', 'fraudulent'],
        num_rows: 10816
    })
    validation: Dataset({
        features: ['title', 'salary_range', 'description', 'required_experience', 'required_education', 'fraudulent'],
        num_rows: 1909
    })
    test: Dataset({
        features: ['title', 'salary_range', 'description', 'required_experience', 'required_education', 'fraudulent'],
        num_rows: 3182
    })
})
Split: train
Columns: ['title', 'salary_range', 'description', 'required_experience', 'required_education', 'fraudulent']


{'title': 'Electrical and Instrumentation Maintenance Technician',
 'salary_range': None,
 'description': 'Maintenance Electrical/Instrumentation TechnicianRed Star Yeast Company LLC (RSYC), a leader in the Yeast Manufacturing Industry, is now accepting resumes for a Maintenance Electrical/Instrumentation (E/I) Technician position at our Cedar Rapids, IA location! RSYC is a joint venture between Lesaffre Yeast Corporation and ADM, and is proud to have a state of the art facility that opened its doors in 2005 as the largest fresh yeast manufacturing facility in North America. The Lesaffre Yeast Corporation has been providing quality yeast products since 1853 and is the world leader in yeast and yeast extracts with a presence in more than 30 countries worldwide.The primary purpose of the Maintenance E/I Technician is to maintain and repair any and all plant electrical equipment and control instruments. Assist in the construction and installation of plant electrical modifications or equip

In [3]:
def combine_text(example):
    text = (
        str(example["title"]) + " " +
        str(example["description"]) + " " +
        str(example["required_experience"]) + " " +
        str(example["required_education"])
    )
    return {"text": text}

ds = ds.map(combine_text)
ds

Map:   0%|          | 0/10816 [00:00<?, ? examples/s]

Map:   0%|          | 0/1909 [00:00<?, ? examples/s]

Map:   0%|          | 0/3182 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'salary_range', 'description', 'required_experience', 'required_education', 'fraudulent', 'text'],
        num_rows: 10816
    })
    validation: Dataset({
        features: ['title', 'salary_range', 'description', 'required_experience', 'required_education', 'fraudulent', 'text'],
        num_rows: 1909
    })
    test: Dataset({
        features: ['title', 'salary_range', 'description', 'required_experience', 'required_education', 'fraudulent', 'text'],
        num_rows: 3182
    })
})

In [11]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\prabh\OneDrive\Desktop\FakeJobBERT\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prabh\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [16]:
from datasets import DatasetDict

# 1) Build text
def combine_text(example):
    return {
        "text": " ".join([
            str(example.get("title", "")),
            str(example.get("description", "")),
            str(example.get("required_experience", "")),
            str(example.get("required_education", "")),
            str(example.get("salary_range", "")),
        ]).strip()
    }

ds2 = DatasetDict({
    "train": ds["train"].map(combine_text),
    "validation": ds["validation"].map(combine_text),
    "test": ds["test"].map(combine_text),
})

# 2) Tokenize
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_ds = ds2.map(tokenize_function, batched=True)

# 3) Labels: force int
def to_int_label(example):
    return {"labels": int(example["fraudulent"])}

tokenized_ds = tokenized_ds.map(to_int_label)

# 4) Remove extra columns so Trainer doesn't get confused
tokenized_ds = tokenized_ds.remove_columns(
    [c for c in tokenized_ds["train"].column_names if c not in ["input_ids", "attention_mask", "labels"]]
)

tokenized_ds.set_format("torch")
tokenized_ds

Map:   0%|          | 0/10816 [00:00<?, ? examples/s]

Map:   0%|          | 0/1909 [00:00<?, ? examples/s]

Map:   0%|          | 0/3182 [00:00<?, ? examples/s]

Map:   0%|          | 0/10816 [00:00<?, ? examples/s]

Map:   0%|          | 0/1909 [00:00<?, ? examples/s]

Map:   0%|          | 0/3182 [00:00<?, ? examples/s]

Map:   0%|          | 0/10816 [00:00<?, ? examples/s]

Map:   0%|          | 0/1909 [00:00<?, ? examples/s]

Map:   0%|          | 0/3182 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10816
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1909
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3182
    })
})

In [18]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",     # ✅ NEW name (fixes your error)
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_ds = ds.map(tokenize_function, batched=True)

Map:   0%|          | 0/10816 [00:00<?, ? examples/s]

Map:   0%|          | 0/1909 [00:00<?, ? examples/s]

Map:   0%|          | 0/3182 [00:00<?, ? examples/s]

In [6]:
tokenized_ds = tokenized_ds.rename_column("fraudulent", "labels")
tokenized_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",          # (new name; works in newer Transformers)
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    compute_metrics=compute_metrics,
)

In [20]:
small_train = tokenized_ds["train"].shuffle(seed=42).select(range(3000))
small_val = tokenized_ds["validation"].shuffle(seed=42).select(range(800))

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics
)

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.159322,0.190337,0.946250,0.000000,0.000000,0.000000


TrainOutput(global_step=188, training_loss=0.16282154397761567, metrics={'train_runtime': 3867.5865, 'train_samples_per_second': 0.776, 'train_steps_per_second': 0.049, 'total_flos': 99350548992000.0, 'train_loss': 0.16282154397761567, 'epoch': 1.0})

In [23]:
trainer.evaluate()

c:\Users\prabh\OneDrive\Desktop\FakeJobBERT\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.19033703207969666,
 'eval_accuracy': 0.94625,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_runtime': 233.3448,
 'eval_samples_per_second': 3.428,
 'eval_steps_per_second': 0.107,
 'epoch': 1.0}

In [24]:
import torch
import numpy as np

def predict_risk(text, threshold=0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).detach().numpy()[0]
    
    risk_score = float(probs[1])  # probability of fake job
    decision = "HIGH_RISK" if risk_score >= threshold else "LOW_RISK"
    
    return {
        "risk_score": risk_score,
        "decision": decision,
        "real_prob": float(probs[0]),
        "fake_prob": float(probs[1])
    }

sample = "Work from home. No experience required. Urgent hiring. Pay registration fee."
predict_risk(sample)

{'risk_score': 0.2311941236257553,
 'decision': 'LOW_RISK',
 'real_prob': 0.7688058018684387,
 'fake_prob': 0.2311941236257553}

In [25]:
model.save_pretrained("./saved_model")
tokenizer.save_pretrained("./saved_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_model\\tokenizer_config.json', './saved_model\\tokenizer.json')

In [27]:
sample = """
We are hiring remote data entry clerks.
No qualifications required.
Send your personal details and pay $50 for equipment processing.
Immediate selection guaranteed.
"""
predict_risk(sample, threshold=0.30)

{'risk_score': 0.3377947509288788,
 'decision': 'HIGH_RISK',
 'real_prob': 0.6622052192687988,
 'fake_prob': 0.3377947509288788}